1. The dataset is different from the air quality one because the columns are a mix of two kinds. Columns like age, chol, thalach, trestbps, and oldpeak are real measurements, but columns like cp, thal, ca, slope, and sex are stored as numbers even though they are really categories. That is why I split them into num_cols and cat_cols and one-hot encoded the category columns, which turns each category into its own 0/1 column. The air quality data was almost all sensor readings, so I didn’t have to worry about this as much. Scaling matters for the SVC because it works by measuring how far apart patients are from each other. chol runs from 126 to 564 and oldpeak only runs from 0 to 6.2, so without scaling, chol would drown out everything else just because its numbers are bigger. The Decision Tree only asks yes or no questions about one column at a time, like whether chol is above a certain number, so the size of the numbers doesn’t change anything and it doesn’t need scaling.

2. Hidden missing values are a problem because pandas doesn’t treat a “?” as missing. It treats it as text, so a number column like ca could turn into a text column, and the scaler or model would either crash or treat the “?” as a real value. In a medical dataset that could mean a blood vessel count or a test result is wrong without anyone noticing. In my data, info() showed ca with 299 values and thal with 301, and when I checked for “?” and counted the blanks, the only missing values were 4 in ca and 2 in thal. For the number columns I used the median, because the values are skewed. chol has a median of 241 but a max of 564, and oldpeak has a median of 0.8 but a max of 6.2, so a few extreme patients would pull the average up and the median stays closer to a typical patient. For the category columns I used the most frequent value, because an average of chest pain types or test results doesn’t mean anything, so filling in the most common category is the safest option. I put both imputers inside the pipeline so the fill values only come from the training data.

3. An SVC with an RBF kernel draws one smooth curved boundary between sick and healthy patients, and which side a new patient lands on depends on how close they are to the patients near that boundary. A Decision Tree instead splits the patients up with a series of yes or no questions, so each patient ends up in a small group and gets the answer that group had. In my results the SVC did better. It got 89% accuracy and a ROC-AUC of 0.964, while the tree got 79% and 0.86. The tree also missed 5 sick patients while the SVC only missed 2, which matters a lot for medical data. The tradeoff is that the tree is easy to explain. I can print the questions it asks and a doctor can follow them step by step, and the feature importances show which tests mattered most. The SVC is a black box. It gives better predictions.

4. C controls how strict the SVC is about getting every training patient right, and gamma controls how closely the boundary hugs the training patients. A high C or a high gamma makes the boundary wiggle around every single patient, which gives low bias but high variance, so it can memorize the training patients and do worse on new ones. A low C or low gamma makes the boundary smoother and simpler, which has higher bias, so it might miss real patterns, but it is steadier on new patients. That is the tradeoff GridSearchCV was balancing. I tested C values of 0.1, 1, 10, and 100 and gamma values of scale, 0.01, 0.1, and 1, and it picked C = 1 and gamma = 0.01, which is a fairly gentle setting and not an extreme one. The cross-validation score was 0.888, and on the test set accuracy went from 89% to 90% and ROC-AUC from 0.964 to 0.968, with false alarms dropping from 5 to 4 and still only 2 sick patients missed. The test score was higher than the cross-validation score.

The tree’s feature importances showed it leaned on three features for about 74% of its decisions. thal_3.0 was the biggest at 0.44, and thal 3 is a normal thallium stress test result, so whether that test comes back normal was the strongest divider between healthy and sick patients. cp_4.0 was next at 0.16, and chest pain type 4 means no chest pain symptoms, then ca_0.0 at 0.14, which means no major blood vessels showing blockage. oldpeak and chol added a smaller amount, and maximum heart rate only got 0.023. For cardiologists, the main thiong is that the stress test result and the vessel scan were more useful than the basic numbers like age or cholesterol. The fact that no chest pain showed up as important means patients without symptoms can still have disease, so screening shouldn’t wait for someone to complain about chest pain. One limit is that this is a single small tree trained on 303 patients, so the exact numbers could change with different data, and it shows which features mattered but doesn’t prove they cause heart disease.